In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-openai
%pip install -Uq python_dotenv

In [ ]:
%pip install -U unstructured-inference

In [ ]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from pathlib import Path

DOCS_DIR = Path("./Docs") 

def partition_all_pdfs(docs_dir: Path = DOCS_DIR):
    """Partition every PDF in the docs folder."""
    pdf_files = sorted(docs_dir.glob("*.pdf"))
    
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in {docs_dir.resolve()}")
    
    all_elements = {}  # file path -> list of elements
    
    for pdf_path in pdf_files:
        print(f"📄 Partitioning: {pdf_path.name}")
        elements = partition_pdf(
            filename=str(pdf_path),
            strategy="hi_res",
            infer_table_structure=True,
            extract_image_block_types=["Image"],
            extract_image_block_to_payload=True,
        )
        all_elements[str(pdf_path)] = elements
        print(f"✅ {pdf_path.name}: {len(elements)} elements")
    
    return all_elements

# Run on every PDF in Docs/
elements_by_file = partition_all_pdfs()

In [ ]:
elements_by_file

In [7]:
print(len(elements_by_file['Docs\\pp_rabi.pdf']))

3520


In [ ]:
images = []
for pdf_key in elements_by_file:
    for element in elements_by_file[pdf_key]:
        if element.category == 'Image':
            images.append(element)

print(f"Found {len(images)} images")
images[10].to_dict()

In [11]:
import base64

d = images[15].to_dict()
img_b64 = d.get("metadata", {}).get("image_base64") 
if not img_b64:
    print("image_base64 not found. available keys:", list(d.keys()))
else:
    with open("image_13.png", "wb") as f:
        f.write(base64.b64decode(img_b64))
    print("Saved image_13.png")

Saved image_13.png


In [ ]:
tables = []
for pdf_key in elements_by_file:
    for element in elements_by_file[pdf_key]:
        if element.category == 'Table':
            tables.append(element)

print(f"Found {len(tables)} tables")
tables[12].to_dict()

In [21]:
def create_chunks_by_title(elements_by_file):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    # Flatten all elements from all PDFs
    elements = []
    for pdf_key in elements_by_file:
        elements.extend(elements_by_file[pdf_key])
    
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2850,
        combine_text_under_n_chars=2000
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements_by_file)

🔨 Creating smart chunks...
✅ Created 326 chunks


In [ ]:
set([str(type(chunk)) for chunk in chunks])
# chunks[50].to_dict()
print(chunks[50])
# chunks[50].metadata.orig_elements[-2].to_dict()

In [ ]:
counter = 0

def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    global counter
    counter += 1
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        
        # Build the text prompt
        prompt_text = (
            "You are creating a searchable, retrieval-focused description for mixed "
            "document content.\n\n"
            "CONTENT TO ANALYZE:\n"
            "TEXT CONTENT:\n"
            f"{text}\n\n"
        )
        
        if tables:
            prompt_text += f"TABLES ({len(tables)}):\n"
            for i, table in enumerate(tables, start=1):
                prompt_text += f"Table {i}:\n{table}\n\n"
        
        if images:
            prompt_text += (
            f"IMAGES ({len(images)}):\n"
            "There are image attachments included. Analyze the visual content as much "
            "as possible from the provided image data and describe likely charts, "
            "diagrams, or visual patterns.\n\n"
            )
        
        prompt_text += (
            "TASK:\n"
            "Generate a detailed, searchable description that covers:\n"
            "1. Key facts, numbers, and data points from the text and tables.\n"
            "2. Main topics, themes, and concepts discussed.\n"
            "3. Questions this content could answer.\n"
            "4. Visual content analysis for images, including likely charts, diagrams, "
            "patterns, or visual relationships.\n"
            "5. Alternative search terms and keywords users might use.\n"
            "6. Important context or document structure cues.\n\n"
            "Do not invent content; rely only on the supplied text, tables, and image "
            "information.\n\n"
            "SEARCHABLE DESCRIPTION:"
        )
        
        message_content = [{"type": "text", "text": prompt_text}]
        
        for image_base64 in images:
            message_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

In [ ]:
p1=processed_chunks
p1 = processed_chunks
print(p1[323].page_content)

In [49]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
# Ensure we have LangChain Document objects for Chroma
if not processed_chunks or not isinstance(processed_chunks[0], Document):
    processed_chunks = summarise_chunks(chunks)

db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [ ]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data


In [ ]:
query = "to plant in South Western region of Punjab, I'm looking for a 6 months improved crop variety, which varietyof PBW can I sow?"
retriever = db.as_retriever(search_kwargs={"k": 5})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

In [ ]:
def rrf_retrieve_documents(vectorstore, query_list, k=3, rrf_k=60):
    """Retrieve documents using reciprocal rank fusion across multiple queries."""
    fused_scores = {}
    for query in query_list:
        results = vectorstore.similarity_search_with_score(query, k=k)
        for rank, (doc, score) in enumerate(results, start=1):
            key = doc.page_content
            if key not in fused_scores:
                fused_scores[key] = {"doc": doc, "score": 0.0}
            fused_scores[key]["score"] += 1.0 / (rrf_k + rank)

    ranked_docs = sorted(fused_scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked_docs][:k]

def build_prompt(question, retrieved_docs, history):
    history_text = ""
    if history:
        history_text = "\n".join(f"User: {q}\nAssistant: {a}" for q, a in history)

    docs_text = ""
    for i, doc in enumerate(retrieved_docs):
        docs_text += f"Document {i + 1}:\n{doc.page_content}\n"
        
        # Extract and include metadata if available
        if hasattr(doc, 'metadata') and 'original_content' in doc.metadata:
            try:
                original_content = json.loads(doc.metadata['original_content'])
                if original_content.get('tables_html'):
                    docs_text += f"Tables in this document:\n"
                    for table in original_content['tables_html']:
                        docs_text += f"{table}\n"
                if original_content.get('images_base64'):
                    docs_text += f"[Contains {len(original_content['images_base64'])} image(s)]\n"
            except (json.JSONDecodeError, KeyError):
                pass
        docs_text += "\n"

    prompt = (
        "You are a retrieval-augmented agricultural assistant. "
        "Answer the user question using only the provided documents and the conversation history.\n\n"
        f"Retrieved documents:\n{docs_text}\n\n"
    )
    if history_text:
        prompt += f"Conversation history:\n{history_text}\n\n"
    prompt += f"Question:\n{question}\n\nAnswer clearly and directly."
    return prompt

def rag_multi_query_chat(questions, history=None, k=3):
    if history is None:
        history = []

    retrieved_docs = rrf_retrieve_documents(db, questions, k=k)

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    answers = []

    for question in questions:
        prompt = build_prompt(question, retrieved_docs, history)
        message = HumanMessage(content=prompt)
        response = llm.invoke([message])
        answer = response.content
        answers.append(answer)
        history.append((question, answer))

    return answers, history

queries = [
    "Which improved PBW variety can I sow in south western Punjab for a 6-month season?",
    "What are the recommended sowing dates and irrigation guidelines for wheat in south western Punjab?",
    "Which pests and diseases should I watch for in wheat grown in south western Punjab?"
]

answers, chat_history = rag_multi_query_chat(queries, k=3)

for i, (question, answer) in enumerate(zip(queries, answers), start=1):
    print(f"--- Question {i} ---")
    print(question)
    print(answer)
    print()
while(1):
    # For manual question input
    manual_question = input("Enter your question: ")
    if(manual_question.strip().lower() in ["exit", "quit"]):
        print("Exiting...")
        break
    manual_answers, chat_history = rag_multi_query_chat([manual_question], history=chat_history, k=3)
    print(f"--- Manual Question ---")
    print(manual_question)
    print(manual_answers[0])

In [ ]:
%pip install -Uq flask

import json
import threading

from flask import Flask, jsonify, request

def generate_similar_questions(question, num_variants=3):
    """Generate alternative phrasings with the same meaning for multi-query retrieval."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    prompt = (
        f"Generate {num_variants} alternative phrasings of this agricultural question "
        "that keep the same meaning but use different words.\n"
        "Return ONLY a JSON array of strings.\n\n"
        f"Question: {question}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        variants = json.loads(response.content.strip())
        if not isinstance(variants, list):
            raise ValueError("Expected a JSON array")
        variants = [str(v).strip() for v in variants if str(v).strip()]
    except (json.JSONDecodeError, ValueError):
        variants = []

    queries = [question]
    for variant in variants:
        if variant.lower() != question.lower() and variant not in queries:
            queries.append(variant)

    return queries[: num_variants + 1]

# Load vector store if the notebook kernel was restarted
if "db" not in globals():
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    db = Chroma(
        persist_directory="dbv1/chroma_db",
        embedding_function=embedding_model,
        collection_metadata={"hnsw:space": "cosine"},
    )

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

@app.route("/ask", methods=["POST"])
def ask():
    data = request.get_json(silent=True) or {}
    question = (data.get("question") or "").strip()

    if not question:
        return jsonify({"error": "question is required"}), 400

    queries = generate_similar_questions(question, num_variants=3)
    answers, _ = rag_multi_query_chat(queries, k=3)

    return jsonify({"answer": answers[0]})

def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

if "flask_thread" not in globals() or not flask_thread.is_alive():
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    print("Flask API running at http://127.0.0.1:5000")
    print('Test with: curl -X POST http://127.0.0.1:5000/ask -H "Content-Type: application/json" -d "{\\"question\\": \\"Which PBW variety is best for south western Punjab?\\"}"')
else:
    print("Flask API is already running at http://127.0.0.1:5000")